# Лабораторная работа 6
**Тема.** 

**Задача 1.**
$$
F_3 | \space | C_{max}
$$ 

Алгоритм: Метод ветвей и границ (4.7)

In [1]:
import copy

tasks = {
    'J1': (1, 13, 6),
    'J2': (10, 12, 18),
    'J3': (17, 9, 13),
    'J4': (12, 17, 2),
    'J5': (11, 3, 5)
}

In [2]:
import copy

def calculate_times(partial_schedule, jobs_data):
    c1, c2, c3 = 0, 0, 0
    for j_id in partial_schedule:
        p1, p2, p3 = jobs_data[j_id]
        c1 += p1
        c2 = max(c1, c2) + p2
        c3 = max(c2, c3) + p3
    return c1, c2, c3

def get_lower_bound(partial_schedule, remaining_ids, jobs_data):
    if not remaining_ids:
        c1, c2, c3 = calculate_times(partial_schedule, jobs_data)
        return c3
    
    c1, c2, c3 = calculate_times(partial_schedule, jobs_data)
    
    sum_p1 = sum(jobs_data[j][0] for j in remaining_ids)
    min_p23 = min(jobs_data[j][1] + jobs_data[j][2] for j in remaining_ids)
    lb1 = c1 + sum_p1 + min_p23
    
    sum_p2 = sum(jobs_data[j][1] for j in remaining_ids)
    min_p3 = min(jobs_data[j][2] for j in remaining_ids)
    lb2 = c2 + sum_p2 + min_p3
    
    sum_p3 = sum(jobs_data[j][2] for j in remaining_ids)
    lb3 = c3 + sum_p3
    
    return max(lb1, lb2, lb3)

best_cmax = float('inf')
best_order = []

def branch_and_bound(current_schedule, remaining_ids, jobs_data):
    global best_cmax, best_order
    
    lb = get_lower_bound(current_schedule, remaining_ids, jobs_data)
    
    if lb >= best_cmax:
        return

    if not remaining_ids:
        _, _, final_c3 = calculate_times(current_schedule, jobs_data)
        if final_c3 < best_cmax:
            best_cmax = final_c3
            best_order = current_schedule
        return

    for i in range(len(remaining_ids)):
        next_job = remaining_ids[i]
        new_remaining = remaining_ids[:i] + remaining_ids[i+1:]
        branch_and_bound(current_schedule + [next_job], new_remaining, jobs_data)

branch_and_bound([], list(tasks.keys()), tasks)

print(f"Точное оптимальное расписание: {best_order}")
print(f"Минимальный Makespan (C_max): {best_cmax}")

Точное оптимальное расписание: ['J1', 'J2', 'J3', 'J4', 'J5']
Минимальный Makespan (C_max): 65


Алгоритм: Campbell-Dudek-Smith (4.6.2)

In [3]:
def solve_johnson_rule(jobs):
    left = []
    right = []
    remaining_jobs = list(jobs)
    
    while remaining_jobs:
        min_val = float('inf')
        best_job = None
        machine = None
        
        for job in remaining_jobs:
            if job['p1'] < min_val:
                min_val = job['p1']
                best_job = job
                machine = 1
            if job['p2'] < min_val:
                min_val = job['p2']
                best_job = job
                machine = 2
        
        if machine == 1:
            left.append(best_job['id'])
        else:
            right.insert(0, best_job['id'])
            
        remaining_jobs.remove(best_job)
        
    optimal_order = left + right
    return optimal_order

def calculate_makespan_f3(order, p1, p2, p3):
    c1, c2, c3 = 0, 0, 0
    for j in order:
        c1 += p1[j]
        c2 = max(c1, c2) + p2[j]
        c3 = max(c2, c3) + p3[j]
    return c3

def solve_f3_cds(jobs):
    p1 = {id: v[0] for id, v in jobs.items()}
    p2 = {id: v[1] for id, v in jobs.items()}
    p3 = {id: v[2] for id, v in jobs.items()}
    
    results = []
    
    j1_data = [{'id': id, 'p1': v[0], 'p2': v[2]} for id, v in jobs.items()]
    order1 = solve_johnson_rule(j1_data)
    res1 = calculate_makespan_f3(order1, p1, p2, p3)
    
    j2_data = [{'id': id, 'p1': v[0]+v[1], 'p2': v[1]+v[2]} for id, v in jobs.items()]
    order2 = solve_johnson_rule(j2_data)
    res2 = calculate_makespan_f3(order2, p1, p2, p3)
    
    if res1 <= res2:
        return order1, res1
    else:
        return order2, res2

best_order, min_cmax = solve_f3_cds(tasks)
print(f"Лучшее расписание по CDS: {best_order}")
print(f"C_max (Makespan) = {min_cmax}")

Лучшее расписание по CDS: ['J1', 'J2', 'J3', 'J4', 'J5']
C_max (Makespan) = 65
